In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE
import pickle
import joblib

from  sklearn.pipeline import Pipeline


In [2]:
pd.set_option('display.max_columns', None)
df = pd.read_excel('../underwriting_50k_dataset.xlsx', sheet_name='Claims_Fraud_Detection')
df.head()

,claim_id,customer_id,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,insured_sex,insured_education_level,insured_occupation,insured_hobbies,insured_relationship,capital_gains,capital_loss,credit_score,telematics_score,incident_date,incident_type,collision_type,incident_severity,authorities_contacted,incident_state,incident_city,incident_location,incident_hour_of_the_day,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,prior_claims_count,fraud_score,fraud_pattern,rule_flag,decline_reason,claim_outcome,submission_text,adjuster_notes,fraud_reported,policy_effective_date,policy_expiration_date,incident_in_policy_period,policy_status_at_incident,incident_near_boundary,is_complex_claim
0,CL-129460,CUST-93810,23,28,POL-651065,2021-06-29,PA,250/500,1000,1262.80,1000000,71927,FEMALE,JD,adm-clerical,exercise,wife,0,9272,496,22.1,2020-07-17,Single Vehicle Collision,Front Collision,Total Loss,Police,PA,North Judithbury,Residential Street,18,4,NO,1,1,?,16915,2884,3359,10672,Chevrolet,Equinox,2011,0,0.0249,NaN,NONE,Policy cancelled prior to loss,Closed – No Payment,"I need to file a claim. On 2020-07-17, I was i...",[2020-08-04] Liability assessment: Based on vi...,N,2021-06-29,2022-06-29,N,Not Yet Effective,N,Y
1,CL-248851,CUST-24592,260,30,POL-272752,2018-06-12,MA,250/500,250,1599.10,0,28722,FEMALE,Associate,handlers-cleaners,camping,not-in-family,0,0,682,50.8,2023-01-27,Single Vehicle Collision,Rear Collision,Minor Damage,NaN,MA,Robinsonshire,Highway,19,3,YES,3,0,YES,55961,284,11360,44317,Kia,Soul,2004,2,0.7113,Staged collision with known associate,PRIOR_CLAIMS,Coverage lapsed at incident date,Denied,Claim submission: Date of loss 2023-01-27. Typ...,[2023-02-23] Medical records received from Ort...,Y,2018-06-12,2019-06-12,N,Expired,N,Y
2,CL-350302,CUST-13278,321,59,POL-851728,2013-02-12,NJ,500/1000,1000,2783.56,0,50116,FEMALE,Bachelors,handlers-cleaners,skydiving,not-in-family,0,21182,685,98.6,2019-06-13,Vehicle Theft,Side Collision,Trivial Damage,Fire,NJ,Curtisfurt,Interstate,18,4,?,0,0,YES,22055,0,924,21131,Kia,Soul,2006,3,0.9183,Vehicle reported stolen but located nearby,PRIOR_CLAIMS,Incident date outside policy period,Denied,"Hi, I'd like to report a Vehicle Theft that oc...",[2019-06-27] Witness interview conducted with ...,Y,2013-02-12,2014-02-12,N,Expired,N,Y
3,CL-965465,CUST-22943,350,55,POL-662583,2012-06-17,NC,500/1000,1000,3111.41,0,27961,MALE,Masters,sales,yoga,husband,0,0,632,34.3,2022-12-06,Vehicle Theft,Rear Collision,Major Damage,Other,NC,New Kellystad,Interstate,9,1,NO,1,2,YES,13539,2513,2404,8622,Hyundai,Sonata,2007,2,0.0504,NaN,NONE,Policy cancelled prior to loss,Denied,"Filing auto claim – 2022-12-06, Vehicle Theft,...",[2022-12-07] Witness interview conducted with ...,N,2012-06-17,2013-06-17,N,Expired,N,Y
4,CL-324958,CUST-42098,112,49,POL-101688,2013-08-29,NY,500/1000,500,3782.20,100000,84387,FEMALE,High School,craft-repair,sleeping,not-in-family,0,0,474,66.4,2024-04-05,Multi-vehicle Collision,Side Collision,Minor Damage,Fire,NY,Jacquelineland,Residential Street,20,2,?,1,0,YES,57783,10918,10288,36577,Jeep,Cherokee,2021,4,0.8127,Inflated repair estimate submitted,SUSPICIOUS_HOUR,Policy cancelled prior to loss,Under Investigation,My car was in an accident on 2024-04-05. It ha...,[2024-05-04] Claimant submitted repair estimat...,Y,2013-08-29,2014-08-29,N,Expired,N,Y


In [3]:
FEATURE_COLUMNS = [
    'months_as_customer',
    'age',
    'insured_sex',
    'insured_education_level',
    'insured_occupation',
    'insured_hobbies',
    'insured_relationship',
    
    'policy_state',
    'policy_csl',
    'policy_deductable',
    'policy_annual_premium',
    'umbrella_limit',
    'capital_gains',
    'capital_loss',
    'credit_score',
    'telematics_score',
    'incident_type',
    'collision_type',
    'incident_severity',
    'authorities_contacted',
    'incident_state',
    'incident_location',
    'incident_hour_of_the_day',
    'number_of_vehicles_involved',
    'property_damage',
    'bodily_injuries',
    'witnesses',
    'police_report_available',
    'total_claim_amount',
    'prior_claims_count',
    'auto_make',
    'auto_year',
    'injury_claim',
    'property_claim',
    'vehicle_claim',
    'incident_in_policy_period',
    'policy_status_at_incident',
    'incident_near_boundary',
    'is_complex_claim',
]
TARGET_COLUMN = 'fraud_reported'
data = df[FEATURE_COLUMNS + [TARGET_COLUMN]]

In [4]:
X = data.drop('fraud_reported', axis=1)
y = (data['fraud_reported'] == 'Y').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=123, 
    stratify=y
)


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class QuestionMarkToNaN(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy() 
        X = X.replace('?', np.nan)
        return X

In [10]:
categorical_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numerical_cols   = X_train.select_dtypes(include=['number']).columns.tolist()

print(f"Numerical cols: {len(numerical_cols)}")
print(f"Categorical cols: {len(categorical_cols)}")

Numerical cols: 19
Categorical cols: 20


In [11]:
numerical_transformer = Pipeline([
    ('qmark', QuestionMarkToNaN()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('qmark', QuestionMarkToNaN()),
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])


In [12]:
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    precision_recall_curve, average_precision_score, f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import (
    GradientBoostingClassifier, RandomForestClassifier,
    AdaBoostClassifier, VotingClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
import os
import tempfile
from sklearn.base import clone

In [13]:
def get_base_models():
    return {
        'gb':  GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=123),
        'rf':  RandomForestClassifier(n_estimators=200, max_depth=20, random_state=123, n_jobs=-1),
        'xgb': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=123, eval_metric='logloss'),
        'lgb': LGBMClassifier(n_estimators=200, max_depth=10, learning_rate=0.1, random_state=123, verbose=-1, n_jobs=-1),
        'lr':  LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=123, class_weight='balanced'),
        'ada': AdaBoostClassifier(n_estimators=200, learning_rate=1.0, random_state=123),
        'mlp': MLPClassifier(hidden_layer_sizes=(100,50), max_iter=200, random_state=123, early_stopping=True)
    }


In [14]:
ensemble_configs = {
    'Top 4 Models (Weighted)': {
        'estimators': ['gb', 'rf', 'xgb', 'lgb'],
        'weights': [1.2, 0.9, 1.0, 1.1]
    },
    'All 7 Models': {
        'estimators': ['gb', 'rf', 'xgb', 'lgb', 'lr', 'ada', 'mlp'],
        'weights': [1.2, 0.9, 1.0, 1.1, 0.7, 0.8, 0.9]
    },
    'Boosting Models Only': {
        'estimators': ['gb', 'xgb', 'lgb', 'ada'],
        'weights': [1.2, 1.0, 1.1, 0.8]
    },
    'Tree-based Models Only': {
        'estimators': ['rf', 'gb', 'xgb', 'lgb'],
        'weights': [0.9, 1.2, 1.0, 1.1]
    },
    'Top 4 Models (Equal)': {
        'estimators': ['gb', 'rf', 'xgb', 'lgb'],
        'weights': [1, 1, 1, 1]
    }
}

In [15]:
from sklearn.base import clone

def make_pipeline(config):
    # Clone the preprocessor (so each ensemble gets its own)
    fresh_preprocessor = clone(preprocessor)
    # Get fresh base models
    base_models = get_base_models()
    estimators = [(name, clone(base_models[name])) for name in config['estimators']]
    ensemble = VotingClassifier(estimators=estimators, voting='soft', weights=config['weights'])
    return ImbPipeline([
        ('preprocessor', fresh_preprocessor),
        ('smote', SMOTE(random_state=123)),
        ('classifier', ensemble)
    ])

In [16]:
results = []
trained_pipelines = {}

print("\n" + "="*60)
for name, cfg in ensemble_configs.items():
    print(f"Training: {name}")
    try:
        pipeline = make_pipeline(cfg)
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        
        auc = roc_auc_score(y_test, y_proba)
        f1 = f1_score(y_test, y_pred)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        recall = tp / (tp + fn)
        precision = tp / (tp + fp)
        
        results.append({
            'Ensemble': name,
            'AUC-ROC': auc,
            'Recall': recall,
            'Precision': precision,
            'F1-Score': f1,
            'Fraud Caught': tp,
            'False Alarms': fp,
            'Fraud Missed': fn
        })
        trained_pipelines[name] = pipeline
        print(f"  AUC: {auc:.4f} | Recall: {recall:.2%} | Precision: {precision:.2%} | F1: {f1:.4f}")
    except Exception as e:
        print(f"  ❌ Error: {e}")


Training: Top 4 Models (Weighted)


/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  AUC: 0.9834 | Recall: 86.05% | Precision: 95.38% | F1: 0.9047
Training: All 7 Models


/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  AUC: 0.9811 | Recall: 85.52% | Precision: 95.56% | F1: 0.9026
Training: Boosting Models Only


...

/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  AUC: 0.9834 | Recall: 86.05% | Precision: 95.38% | F1: 0.9047
Training: Top 4 Models (Equal)


/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  AUC: 0.9833 | Recall: 85.88% | Precision: 95.61% | F1: 0.9048


/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [17]:
results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
print("\n" + "="*80)
print("ENSEMBLE PERFORMANCE COMPARISON")
print(results_df[['Ensemble', 'AUC-ROC', 'Recall', 'Precision', 'F1-Score']].to_string(index=False))

best_name = results_df.iloc[0]['Ensemble']
best_pipeline = trained_pipelines[best_name]
print(f"\n🏆 BEST ENSEMBLE: {best_name} (AUC: {results_df.iloc[0]['AUC-ROC']:.4f})")


ENSEMBLE PERFORMANCE COMPARISON
               Ensemble  AUC-ROC   Recall  Precision  F1-Score
   Boosting Models Only 0.984098 0.867899   0.944478  0.904570
Top 4 Models (Weighted) 0.983429 0.860452   0.953837  0.904741
 Tree-based Models Only 0.983429 0.860452   0.953837  0.904741
   Top 4 Models (Equal) 0.983298 0.858798   0.956095  0.904838
           All 7 Models 0.981084 0.855212   0.955624  0.902634

🏆 BEST ENSEMBLE: Boosting Models Only (AUC: 0.9841)


In [18]:
y_proba_best = best_pipeline.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_best)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_thresh = thresholds[np.argmax(f1_scores)]
best_f1 = f1_scores[np.argmax(f1_scores)]

print(f"\nOptimal threshold (max F1): {best_thresh:.4f}")
print(f"  Recall:    {recalls[np.argmax(f1_scores)]:.2%}")
print(f"  Precision: {precisions[np.argmax(f1_scores)]:.2%}")
print(f"  F1-Score:  {best_f1:.4f}")

# Apply tuned threshold
y_pred_tuned = (y_proba_best >= best_thresh).astype(int)
tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred_tuned).ravel()
print(f"\n  Fraud caught:  {tp_t} / {tp_t + fn_t}")
print(f"  False alarms:  {fp_t}")
print(f"  Fraud missed:  {fn_t}")


/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Optimal threshold (max F1): 0.5252
  Recall:    86.40%
  Precision: 95.34%
  F1-Score:  0.9065

  Fraud caught:  3133 / 3626
  False alarms:  153
  Fraud missed:  493


In [22]:
save_payload = {'pipeline': best_pipeline, 'threshold': best_thresh}
joblib.dump(save_payload, 'best_ensemble_fraud_model.pkl')
print("\n✅ Model + threshold saved to 'best_ensemble_fraud_model.pkl'")



✅ Model + threshold saved to 'best_ensemble_fraud_model.pkl'


In [20]:
X_test['calculated_total'] = X_test['vehicle_claim'] + X_test['injury_claim'] + X_test['property_claim']

loaded = joblib.load('best_ensemble_fraud_model.pkl')
loaded_pipe = loaded['pipeline']
loaded_thresh = loaded['threshold']
y_proba_loaded = loaded_pipe.predict_proba(X_test)[:, 1]
y_pred_loaded = (y_proba_loaded >= loaded_thresh).astype(int)
auc_loaded = roc_auc_score(y_test, y_proba_loaded)
f1_loaded = f1_score(y_test, y_pred_loaded)
print(f"Verification – AUC: {auc_loaded:.4f}, F1: {f1_loaded:.4f}, Threshold: {loaded_thresh:.4f}")

/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Verification – AUC: 0.9840, F1: 0.9044, Threshold: 0.4772


In [21]:
def predict_fraud(X_new, pipeline, threshold=0.5):
    proba = pipeline.predict_proba(X_new)[:, 1]
    flags = (proba >= threshold).astype(int)
    return pd.DataFrame({'fraud_probability': proba, 'fraud_flag': flags}, index=X_new.index)

# Example on first 10 test rows
sample = predict_fraud(X_test.head(10), best_pipeline, threshold=best_thresh)
print("\nSample predictions (first 10 test rows):")
print(sample)

print("\n✅ All steps completed successfully – pipeline is production-ready.")


Sample predictions (first 10 test rows):
       fraud_probability  fraud_flag
33024           0.934917           1
41169           0.130816           0
46596           0.228887           0
48097           0.157508           0
20689           0.058902           0
1257            0.935801           1
41816           0.095845           0
42961           0.317164           0
13190           0.099824           0
31585           0.100990           0

✅ All steps completed successfully – pipeline is production-ready.


/home/lang-chain/Documents/mcp_insurance/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
